In [ ]:
import numpy as np
from xso.parscans import run_xso_parscan, run_xso_stabilityscan

# --- 1. Run the IVP Scan ---
print("Running IVP scan...")
ivp_param_values = np.linspace(0.005, 0.5, 10)
ivp_param_values2 = np.linspace(0.005, 0.1, 10)

ivp_results = run_xso_parscan(
    model_file_name='Stocketal2008',
    model_name='model',
    model_setup_name='model_setup_ivp', # <-- Your IVP setup
    param_name='Outflow__rate',
    param_values=ivp_param_values,
    param_name2='Growth__mu_max',
    param_values2=ivp_param_values2,
    processes=20
)

if ivp_results is None:
    raise RuntimeError("IVP scan failed. Aborting.")

# --- 2. Calculate Mean of Last 1000 Steps ---
print("Calculating mean values for initial conditions...")
# Select variables and get the mean of the last 1000 time steps
mean_results = ivp_results[['Phytoplankton__value', 'Nutrient__value']] \
                         .isel(time=slice(-1000, None)) \
                         .mean(dim='time')

print(mean_results)


# --- 3. Define the Initial Value Mapping ---
# Maps: {Variable name in mean_results} -> {Parameter name in model_setup_stability}
iv_map = {
    'Phytoplankton__value': 'Phytoplankton__value_init'
}

# --- 4. Run the Stability Scan ---
print("Running stability scan with dynamic initial values...")
stability_results = run_xso_stabilityscan(
    model_file_name='Stocketal2008',
    model_setup_name='model_setup_stability', # <-- Your stability setup
    param_name='Outflow__rate',
    param_values=ivp_param_values,       # Must be the same parameter range
    param_name2='Growth__mu_max',
    param_values2=ivp_param_values2,     # Must be the same parameter range
    processes=20,
    initial_values_ds=mean_results,      # <-- Pass the mean results
    iv_mapping=iv_map                    # <-- Pass the name mapping
)

# Display the final result
if stability_results is not None:
    print("\nFinal Stability Scan Output Dataset:")
    print(stability_results)
    print("\nStability data:")
    print(stability_results['stability'])